# Overview

This case study focuses on evaluation of tokenization strategies of noisy texts. This study is split onto two stages: 
1. Modeling OCR errors
2. Tokenization and evaluation


# Stage 1: modeling OCR errors

## Motivation
To evaluate tokenizers on noisy text, we need a way to generate realistic OCR noise. Instead of random perturbations, we model actual statistical distributions found in paired "Clean vs. OCR" datasets. This allows us to simulate physical document degradation and font-specific errors accurately.

## Methods
Distrubutions are learned using BLN_600 dataset containing newspapers with GT (clean) and OCR predicts (corrupted) texts

The error modeling is based on the **Weighted Edit Distance** (Levenshtein) framework. We treat the OCR process as a noisy channel where a clean string is transformed via three operations:

1. **Alignment**: We use the Levenshtein algorithm to map characters between ground truth and OCR text.
2. **Probability Estimation**: We calculate transition probabilities $P(c'|c)$ for substitutions, insertions, and deletions. 
3. **Laplace Smoothing**: To handle unseen transitions, we apply a smoothing factor $\alpha$:
   $$P(c'|c) = \frac{count(c \to c') + \alpha}{\sum_{x} (count(c \to x) + \alpha)}$$
4. **Synthetic Corrupter**: Based on these distributions, we build a generator that applies insertions, deletions, and substitutions to clean text to mimic OCR output.


## Data preparation

In [67]:
from pathlib import Path
from collections import Counter
import Levenshtein
import numpy as np
import pandas as pd
import os
from tqdm import tqdm

In [16]:
OCR_DATA_PATH = Path("data/bln600")

def clean_text(text: str):
    text = text.replace("\n", " ")
    text = " ".join(text.split())
    return text

def load_text_pairs(base_path: Path):
    ocr_dir = base_path / "OCR Text"
    gt_dir = base_path / "Ground Truth"
    pairs = []
    for gt_file in sorted(gt_dir.glob("*.txt")):
        ocr_file = ocr_dir / gt_file.name
        if not ocr_file.exists():
            continue
        gt_text = clean_text(gt_file.read_text(encoding="utf-8", errors="ignore"))
        ocr_text = clean_text(ocr_file.read_text(encoding="utf-8", errors="ignore"))
        pairs.append((gt_file.name, gt_text, ocr_text))
    return pairs

pairs = load_text_pairs(OCR_DATA_PATH)
print(f"loaded_pairs={len(pairs)}")
print("sample_file=", pairs[0][0] if pairs else None)
print("sample_gt_len=", len(pairs[0][1]) if pairs else None)
print("sample_ocr_len=", len(pairs[0][2]) if pairs else None)

loaded_pairs=600
sample_file= 3200797029.txt
sample_gt_len= 2124
sample_ocr_len= 2149


In [17]:
EPS = "<eps>"

def align_with_editops(gt: str, ocr: str):
    ops = Levenshtein.editops(gt, ocr)
    aligned = []
    gt_i = 0
    ocr_i = 0
    op_i = 0

    while gt_i < len(gt) and ocr_i < len(ocr):
        if op_i < len(ops):
            op, src, dst = ops[op_i]
            if op == "replace" and src == gt_i and dst == ocr_i:
                aligned.append((gt[gt_i], ocr[ocr_i]))
                gt_i += 1
                ocr_i += 1
                op_i += 1
                continue
            if op == "delete" and src == gt_i and dst == ocr_i:
                aligned.append((gt[gt_i], EPS))
                gt_i += 1
                op_i += 1
                continue
            if op == "insert" and src == gt_i and dst == ocr_i:
                aligned.append((EPS, ocr[ocr_i]))
                ocr_i += 1
                op_i += 1
                continue
        aligned.append((gt[gt_i], ocr[ocr_i]))
        gt_i += 1
        ocr_i += 1

    while gt_i < len(gt):
        aligned.append((gt[gt_i], EPS))
        gt_i += 1

    while ocr_i < len(ocr):
        aligned.append((EPS, ocr[ocr_i]))
        ocr_i += 1

    return aligned

def train_weighted_levenshtein_channel(pairs, alpha=0.05, weights=(1, 1, 1)):
    counts = Counter()
    weighted_distances = []

    for _, gt, ocr in pairs:
        path = align_with_editops(gt, ocr)
        counts.update(path)
        weighted_distances.append(Levenshtein.distance(gt, ocr, weights=weights))

    total = sum(counts.values()) + alpha * (len(counts) + 1)
    channel_probs = {k: (v + alpha) / total for k, v in counts.items()}
    channel_probs[("<unk>", "<unk>")] = alpha / total

    avg_weighted_distance = sum(weighted_distances) / max(len(weighted_distances), 1)
    return channel_probs, avg_weighted_distance

channel_probs, avg_weighted_distance = train_weighted_levenshtein_channel(
    pairs,
    alpha=0.05,
    weights=(1, 1, 0.5)
    )
print(f"transitions={len(channel_probs)}")
print(f"avg_weighted_distance={avg_weighted_distance:.3f}")

transitions=3826
avg_weighted_distance=65.577


In [18]:
sub_probs = {k: v for k, v in channel_probs.items() if k[0] != EPS and k[1] != EPS and k[0] != "<unk>" and k[1] != "<unk>" and k[0] != k[1]}
ins_probs = {k: v for k, v in channel_probs.items() if k[0] == EPS and k[1] != "<unk>"}
del_probs = {k: v for k, v in channel_probs.items() if k[1] == EPS and k[0] != "<unk>"}

top_sub = sorted(sub_probs.items(), key=lambda x: x[1], reverse=True)[:20]
top_ins = sorted(ins_probs.items(), key=lambda x: x[1], reverse=True)[:20]
top_del = sorted(del_probs.items(), key=lambda x: x[1], reverse=True)[:20]

print("top_substitutions")
for (c_in, c_out), p in top_sub:
    print(f"{c_in}->{c_out}: {p:.6f}")

print("\ntop_insertions")
for (_, c_out), p in top_ins:
    print(f"<eps>->{c_out}: {p:.6f}")

print("\ntop_deletions")
for (c_in, _), p in top_del:
    print(f"{c_in}-><eps>: {p:.6f}")

top_substitutions
 ->-: 0.000902
h->b: 0.000510
s->e: 0.000357
e->o: 0.000356
n->i: 0.000331
m->n: 0.000328
.->,: 0.000325
c->e: 0.000299
o->e: 0.000283
 ->': 0.000283
n->u: 0.000277
h->i: 0.000259
,->.: 0.000255
u->a: 0.000253
c->o: 0.000249
a->e: 0.000240
—->-: 0.000210
u->n: 0.000206
s->a: 0.000195
’->': 0.000194

top_insertions
<eps>-> : 0.006163
<eps>->-: 0.002221
<eps>->i: 0.001739
<eps>->.: 0.001453
<eps>->l: 0.001260
<eps>->': 0.001218
<eps>->t: 0.001172
<eps>->e: 0.000987
<eps>->r: 0.000870
<eps>->n: 0.000807
<eps>->a: 0.000738
<eps>->,: 0.000735
<eps>->s: 0.000732
<eps>->I: 0.000657
<eps>->o: 0.000565
<eps>->h: 0.000345
<eps>->d: 0.000287
<eps>->c: 0.000278
<eps>->1: 0.000275
<eps>->f: 0.000260

top_deletions
 -><eps>: 0.002419
_-><eps>: 0.001408
e-><eps>: 0.000761
n-><eps>: 0.000580
s-><eps>: 0.000537
—-><eps>: 0.000529
a-><eps>: 0.000467
t-><eps>: 0.000453
,-><eps>: 0.000442
i-><eps>: 0.000400
o-><eps>: 0.000394
r-><eps>: 0.000365
h-><eps>: 0.000343
l-><eps>: 0.000318
.-><e

In [19]:
def top_k_for_coverage(prob_dict, coverage=0.9):
    items = sorted(prob_dict.items(), key=lambda x: x[1], reverse=True)
    probs = np.array([v for _, v in items], dtype=float)

    total = probs.sum()
    target = coverage * total

    cumsum = np.cumsum(probs)
    k = int(np.searchsorted(cumsum, target, side="left")) + 1
    selected = items[:k]
    covered = float(cumsum[k - 1])
    covered_ratio = covered / total if total > 0 else 0.0
    return selected, covered

top_sub_90, sub_cum = top_k_for_coverage(sub_probs, coverage=0.9)
top_ins_90, ins_cum = top_k_for_coverage(ins_probs, coverage=0.9)
top_del_90, del_cum = top_k_for_coverage(del_probs, coverage=0.9)

print(f"sub: K={len(top_sub_90)}, cum_prob={sub_cum:.6f}")
print(f"ins: K={len(top_ins_90)}, cum_prob={ins_cum:.6f}")
print(f"del: K={len(top_del_90)}, cum_prob={del_cum:.6f}")

sub: K=1139, cum_prob=0.024427
ins: K=28, cum_prob=0.024347
del: K=25, cum_prob=0.011368


In [20]:
top_sub_90

[((' ', '-'), 0.000902017363941772),
 (('h', 'b'), 0.0005097985044419645),
 (('s', 'e'), 0.0003572689479698172),
 (('e', 'o'), 0.00035612210919935),
 (('n', 'i'), 0.00033089165624906994),
 (('m', 'n'), 0.00032802455932290174),
 (('.', ','), 0.0003251574623967336),
 (('c', 'e'), 0.0002993535900612199),
 (('o', 'e'), 0.0002827244278894445),
 ((' ', "'"), 0.0002827244278894445),
 (('n', 'u'), 0.0002769902340371081),
 (('h', 'i'), 0.00025864081370963177),
 ((',', '.'), 0.00025520029739823),
 (('u', 'a'), 0.00025290661985729544),
 (('c', 'o'), 0.00024889268416065994),
 (('a', 'e'), 0.00023971797399692177),
 (('—', '-'), 0.00020990016596477267),
 (('u', 'n'), 0.00020645964965337085),
 (('s', 'a'), 0.00019499126194869813),
 (('’', "'"), 0.00019441784256346448),
 (('n', 'a'), 0.0001760684222359881),
 (('n', 'l'), 0.00017377474469505355),
 (('h', 'l'), 0.00017377474469505355),
 (('a', 'i'), 0.00016689371207224992),
 (('a', 's'), 0.00016230635699038083),
 (('b', 'h'), 0.0001600126794494463),
 ((

## Text corruptor

In [21]:
class TextCorruptor:
    def __init__(self, top_sub, top_ins, top_del, seed=None, insert_rate=0.03, delete_rate=0.03, sub_rate=0.04):
        self.rng = np.random.default_rng(seed)
        self.insert_rate = insert_rate
        self.delete_rate = delete_rate
        self.sub_rate = sub_rate
        self.sub_map = {}
        self.ins_chars = []
        self.ins_probs = np.array([], dtype=float)
        self.del_chars = []
        self.del_probs = np.array([], dtype=float)

        sub_items = [(src, dst, prob) for (src, dst), prob in top_sub if src != '<eps>' and dst != '<eps>' and src != dst]
        for src, dst, prob in sub_items:
            self.sub_map.setdefault(src, []).append((dst, prob))
        for src in list(self.sub_map.keys()):
            items = self.sub_map[src]
            chars = [dst for dst, _ in items]
            probs = np.array([prob for _, prob in items], dtype=float)
            self.sub_map[src] = (chars, probs / probs.sum())

        ins_items = [(dst, prob) for (src, dst), prob in top_ins if src == '<eps>']
        if ins_items:
            self.ins_chars = [dst for dst, _ in ins_items]
            self.ins_probs = np.array([prob for _, prob in ins_items], dtype=float)
            self.ins_probs = self.ins_probs / self.ins_probs.sum()

        del_items = [(src, prob) for (src, dst), prob in top_del if dst == '<eps>']
        if del_items:
            self.del_chars = [src for src, _ in del_items]
            self.del_probs = np.array([prob for _, prob in del_items], dtype=float)
            self.del_probs = self.del_probs / self.del_probs.sum()

    def sample_insertion(self):
        if not self.ins_chars:
            return ''
        return self.rng.choice(self.ins_chars, p=self.ins_probs)

    def sample_deletion(self, ch):
        if not self.del_chars:
            return False
        if ch not in self.del_chars:
            return False
        idx = self.del_chars.index(ch)
        prob = self.del_probs[idx]
        scale = prob / self.del_probs.max() if self.del_probs.max() > 0 else 0.0
        return self.rng.random() < self.delete_rate * scale

    def sample_substitution(self, ch):
        if ch not in self.sub_map:
            return ch
        chars, probs = self.sub_map[ch]
        if len(chars) == 0:
            return ch
        return self.rng.choice(chars, p=probs)

    def corrupt(self, text):
        out = []
        for ch in text:
            if self.ins_chars and self.rng.random() < self.insert_rate:
                out.append(self.sample_insertion())
            if self.sample_deletion(ch):
                continue
            if self.rng.random() < self.sub_rate:
                out.append(self.sample_substitution(ch))
            else:
                out.append(ch)
            if self.ins_chars and self.rng.random() < self.insert_rate:
                out.append(self.sample_insertion())
        return ''.join(out)

corruptor = TextCorruptor(
    top_sub_90,
    top_ins_90,
    top_del_90,
    insert_rate=0.07,
    delete_rate=0.07,
    sub_rate=0.08,
    seed=42
)
sample_text = pairs[0][1] if pairs else ''
print(corruptor.corrupt(sample_text[:200]))
print(sample_text[:200])

ROBBERIY &T A BAeONETS..- ED WARD PRING, ltwentyI-se'ven., carpeter, wasbrougtht-tup  ob remand at the Greenwich police- c6u rt,tcharged with steaIlin  jewallcry tof -the va,lueiof 'i£100, tho. property ba.lr slir R.e heort C
ROBBERY AT A BARONETS. EDWARD PRING, twenty-seven, carpenter, was brought up on remand at the Greenwich Police-court, charged with stealing jewellery to the value of £100, the property of Sir Robert C


# Stage 2: tokenization, intrinsic & extrinsic evaluation

To compare tokenization approaches following intrinsic metrics are used:
1. Compression Ratio $compression\_ratio = \frac{\# total\_chars}{\# tokens\_in\_text}$
2. $fertility=\frac{\# total\_tokens}{\# total\_words}$
3. $fragmentation\_rate = \frac{\# split\_words}{\# total\_words}$
4. $stability\_under\_noise = 1 - \frac{EditDistance(T_{clean}, T_{noisy})}{max(|T_{clean}|, |T_{noisy}|)}$


As a downstream task we choose newspaper text classification on [AG_news](https://www.kaggle.com/datasets/amananandrai/ag-news-classification-dataset?select=train.csv) dataset using pretrained [roBERTa](https://huggingface.co/textattack/roberta-base-ag-news) model.


## Methods

AG News clean dataset are corrupted using methods defined in stage 1. Then model's embedding and head layers are learned with new tokenizer to adapt embeddings for it, rest of the layers are frozen. Finally model is uvaluated with different tokenizers on a test set, f1 and accuracy are used as extrinsic metrics.


## Baseline (out of the box tokenizer)

In [7]:
CLASSIFICATION_DATA_PATH = Path("data/ag_news")

train_df = pd.read_csv(
    os.path.join(CLASSIFICATION_DATA_PATH, "train.csv")
)

test_df = pd.read_csv(
    os.path.join(CLASSIFICATION_DATA_PATH, "test.csv")
)

train_df.head()

,Class Index,Title,Description
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli..."
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...
3,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...
4,3,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco..."


In [8]:
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score

In [95]:
MODEL_NAME = 'textattack/roberta-base-AG-news'
LABEL_COL = 'Class Index'
TEXT_COLS = [c for c in train_df.columns if c != LABEL_COL]
device = "cuda" if torch.cuda.is_available() else "cpu"

base_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Running on {device}")

Running on cuda


In [ ]:
corruptor = TextCorruptor(
    top_sub_90,
    top_ins_90,
    top_del_90,
    insert_rate=0.07,
    delete_rate=0.07,
    sub_rate=0.08,
    seed=42
)

train_df_corrupted = train_df.copy()
test_df_corrupted = test_df.copy()

for col in TEXT_COLS:
    train_df_corrupted[col] = train_df_corrupted[col].apply(
        lambda x: corruptor.corrupt(str(x))
    )
    test_df_corrupted[col] = test_df_corrupted[col].apply(
        lambda x: corruptor.corrupt(str(x))
    )

In [36]:
def build_text_frame(df):
    frame = df.copy()
    if len(TEXT_COLS) == 1:
        frame['text'] = frame[TEXT_COLS[0]].astype(str)
    else:
        frame['text'] = frame[TEXT_COLS].astype(str).agg(' '.join, axis=1)
    frame['label'] = frame[LABEL_COL].astype(int) - 1
    return frame[['text', 'label']]

train_data = build_text_frame(train_df)
test_data = build_text_frame(test_df)

train_data_corrupted = build_text_frame(
    train_df_corrupted
)
test_data_corrupted = build_text_frame(
    test_df_corrupted
)

In [ ]:
class TextClassificationDataset(Dataset):
    def __init__(self, frame, tokenizer, max_length=128):
        self.texts = frame['text'].tolist()
        self.labels = frame['label'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding=False,
            return_tensors=None
        )
        encoded['labels'] = self.labels[idx]
        return encoded

train_dataset = TextClassificationDataset(train_data, base_tokenizer)
test_dataset = TextClassificationDataset(test_data, base_tokenizer)


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4
).to(device)

data_collator = DataCollatorWithPadding(tokenizer=base_tokenizer)

training_args = TrainingArguments(
    output_dir='roberta_ag_news_classifier',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=100,
    save_strategy='no',
    report_to='none',
    fp16=torch.cuda.is_available()
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'macro_f1': f1_score(labels, preds, average='macro')
    }

trainer = Trainer(
    model=model,
    args=training_args,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: textattack/roberta-base-AG-news
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
print("Clean texts baseline evaluation:")

pred_output = trainer.predict(test_dataset)

baseline_test_preds = pred_output.predictions.argmax(axis=-1)
baseline_test_labels = pred_output.label_ids

print("test_accuracy=", accuracy_score(baseline_test_labels, baseline_test_preds))
print("test_macro_f1=", f1_score(baseline_test_labels, baseline_test_preds, average="macro"))

{'test_loss': 0.1854841709136963, 'test_model_preparation_time': 0.0057, 'test_accuracy': 0.9469736842105263, 'test_macro_f1': 0.9469397660985941, 'test_runtime': 9.2136, 'test_samples_per_second': 824.868, 'test_steps_per_second': 51.554}
test_accuracy= 0.9469736842105263
test_macro_f1= 0.9469397660985941


In [ ]:
print("Corrupted texts evaluation:")

test_dataset_corrupted = TextClassificationDataset(test_data_corrupted, base_tokenizer)

pred_output = trainer.predict(test_dataset_corrupted)

baseline_test_preds = pred_output.predictions.argmax(axis=-1)
baseline_test_labels = pred_output.label_ids

print("test_accuracy=", accuracy_score(baseline_test_labels, baseline_test_preds))
print("test_macro_f1=", f1_score(baseline_test_labels, baseline_test_preds, average="macro"))

Corrupted texts evaluation:


test_accuracy= 0.5628947368421052
test_macro_f1= 0.563574251206808


## Tokenizers

Three tokenization approaches are used in this study:
1. Byte level BPE with dropout
2. Sentence Piece
3. [LiteToken](https://arxiv.org/abs/2602.04706) (IMR pruning for BPE)

## Dropout BPE

**Idea**: Randomly removes merges during training with a probability $p$. This forces the model to see different granularities of the same word, preventing it from over-relying on a single character that might be corrupted by OCR.

In [58]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, normalizers, decoders
from transformers import PreTrainedTokenizerFast


In [90]:
TOKENIZER_OUTPUT_DIR = Path("tokenizers/dropout_bpe")
TOKENIZER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BPE_DROPOUT = 0.3
BPE_VOCAB_SIZE = 50000
SPECIAL_TOKENS = ["<pad>", "<unk>", "<mask>", "<s>", "</s>"]

corrupted_train_texts = [corruptor.corrupt(text) for text in train_data["text"].astype(str).tolist()]

dropout_bpe = Tokenizer(models.BPE(unk_token="<unk>", dropout=BPE_DROPOUT))
dropout_bpe.normalizer = normalizers.Sequence([normalizers.NFKC()])
dropout_bpe.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)
bpe_trainer = trainers.BpeTrainer(vocab_size=BPE_VOCAB_SIZE, special_tokens=SPECIAL_TOKENS)
dropout_bpe.train_from_iterator(corrupted_train_texts, trainer=bpe_trainer)
dropout_bpe.save(str(TOKENIZER_OUTPUT_DIR / "tokenizer.json"))

In [48]:
dropout_bpe_tokenizer = PreTrainedTokenizerFast(
    tokenizer_file=str(TOKENIZER_OUTPUT_DIR / "tokenizer.json"),
    unk_token="<unk>",
    pad_token="<pad>",
    cls_token="<s>",
    sep_token="</s>",
    mask_token="<mask>",
    bos_token="<s>",
    eos_token="</s>"
 )

print("dropout_bpe_vocab_size", len(dropout_bpe_tokenizer))
print("corrupted_train_samples", len(corrupted_train_texts))

dropout_bpe_vocab_size 50000
corrupted_train_samples 120000


## SentencePiece (Unigram)

**Idea**: Treats tokenization as a probabilistic task, starting with a large vocabulary and pruning it. It can provide multiple possible segmentations (subword regularization), making models more robust to character-level noise.

Maximizes the likelihood of the sequence $X$:
$$P(X) = \prod_{i=1}^{n} P(x_i)$$

In [59]:
SENTENCEPIECE_OUTPUT_DIR = Path("tokenizers/sentencepiece_unigram")
SENTENCEPIECE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SP_VOCAB_SIZE = 50000

sentencepiece_unigram = Tokenizer(models.Unigram())
sentencepiece_unigram.normalizer = normalizers.Sequence([normalizers.NFKC()])
sentencepiece_unigram.pre_tokenizer = pre_tokenizers.Metaspace(replacement="_", prepend_scheme="always")
sentencepiece_unigram.decoder = decoders.Metaspace(replacement="_", prepend_scheme="always")

sp_trainer = trainers.UnigramTrainer(
    vocab_size=SP_VOCAB_SIZE,
    special_tokens=SPECIAL_TOKENS,
    unk_token="<unk>"
 )

sentencepiece_unigram.train_from_iterator(corrupted_train_texts, trainer=sp_trainer)
sentencepiece_unigram.save(str(SENTENCEPIECE_OUTPUT_DIR / "tokenizer.json"))

sentencepiece_tokenizer = PreTrainedTokenizerFast(
    tokenizer_file=str(SENTENCEPIECE_OUTPUT_DIR / "tokenizer.json"),
    unk_token="<unk>",
    pad_token="<pad>",
    cls_token="<s>",
    sep_token="</s>",
    mask_token="<mask>",
    bos_token="<s>",
    eos_token="</s>"
 )

print("sentencepiece_vocab_size", len(sentencepiece_tokenizer))



sentencepiece_vocab_size 50000

sentencepiece_vocab_size 50000


## LiteToken (IMR pruning for BPE)

LiteToken is a method to prune **Intermediate Merge Residues** from BPE tokenizers to improve efficiency and robustness.


**idea:** BPE creates many tokens that are later "covered up" by longer merges. These intermediate tokens stay in the vocabulary but are rarely used, wasting parameters and computational FLOPs. Tokens that are frequent during merge learning so that retained in the final vocabulary, but are mostly further merged and rarely emitted when tokenizing the corpus during tokenizer usage. Such low-frequency tokens not only waste vocabulary capacity but also increase vulnerability to adversarial or atypical inputs (such as OCR)

### 2. The LiteToken Process
1. **FI Ratio Calculation**: Measure the actual emission frequency vs. merge-time frequency.
2. **Entropy Check**: Filter tokens to keep meaningful roots/suffixes even if they have low usage.
3. **Pruning**: Remove the "Red Tokens" (Low usage + Low entropy).
4. **Plug-and-Play**: Use the new tokenizer with the original model (no fine-tuning required).

In [91]:
import json
import math
from collections import defaultdict

LITETOKEN_OUTPUT_DIR = Path("tokenizers/litetoken_pruned")
LITETOKEN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LITETOKEN_RATIO_THRESHOLD = 0.7
LITETOKEN_ENTROPY_THRESHOLD = 5.0

with open(TOKENIZER_OUTPUT_DIR / "tokenizer.json", "r", encoding="utf-8") as f:
    bpe_json = json.load(f)

base_vocab = bpe_json["model"]["vocab"]
base_merges = bpe_json["model"]["merges"]

id_to_token = {idx: tok for tok, idx in base_vocab.items()}
special_token_set = set(SPECIAL_TOKENS)

freq_counter = Counter()
left_neighbors = defaultdict(Counter)
right_neighbors = defaultdict(Counter)

for text in corrupted_train_texts + train_data["text"].astype(str).tolist():
    ids = dropout_bpe_tokenizer.encode(str(text), add_special_tokens=False)
    seq_tokens = [id_to_token.get(token_id, "<unk>") for token_id in ids]
    freq_counter.update(seq_tokens)
    for i, token in enumerate(seq_tokens):
        if i > 0:
            left_neighbors[token][seq_tokens[i - 1]] += 1
        if i + 1 < len(seq_tokens):
            right_neighbors[token][seq_tokens[i + 1]] += 1

children_map = defaultdict(list)
for merge_rule in base_merges:
    if isinstance(merge_rule, str):
        left, right = merge_rule.split(" ")
    else:
        left, right = merge_rule
    merged = left + right
    children_map[left].append(merged)
    children_map[right].append(merged)

descendant_freq_memo = {}

def descendant_freq(token):
    if token in descendant_freq_memo:
        return descendant_freq_memo[token]
    total = 0
    for child in children_map.get(token, []):
        child_freq = freq_counter.get(child, 0)
        total += child_freq + descendant_freq(child)
    descendant_freq_memo[token] = total
    return total

def neighbor_entropy(counter_obj):
    total = sum(counter_obj.values())
    if total == 0:
        return 0.0
    entropy = 0.0
    for count in counter_obj.values():
        prob = count / total
        entropy -= prob * math.log(prob + 1e-12)
    return entropy

candidate_tokens = []
for token in base_vocab.keys():
    if token in special_token_set:
        continue
    final_freq = freq_counter.get(token, 0)
    interm_freq = descendant_freq(token)
    denom = interm_freq + final_freq
    fi_ratio = final_freq / denom if denom > 0 else 1.0
    left_h = neighbor_entropy(left_neighbors[token])
    right_h = neighbor_entropy(right_neighbors[token])
    entropy_score = min(left_h, right_h)
    if fi_ratio <= LITETOKEN_RATIO_THRESHOLD and entropy_score <= LITETOKEN_ENTROPY_THRESHOLD:
        candidate_tokens.append(token)

candidate_tokens = [token for token in candidate_tokens if len(token.strip()) > 1 and token not in special_token_set]
imr_set = set(candidate_tokens)

pruned_tokens = [token for token, _ in sorted(base_vocab.items(), key=lambda x: x[1]) if token not in imr_set]
pruned_vocab = {token: new_id for new_id, token in enumerate(pruned_tokens)}

pruned_merges = []
for merge_rule in base_merges:
    if isinstance(merge_rule, str):
        left, right = merge_rule.split(" ")
    else:
        left, right = merge_rule
    merged = left + right
    if left in pruned_vocab and right in pruned_vocab and merged in pruned_vocab:
        pruned_merges.append((left, right))

litetoken_core = Tokenizer(models.BPE(vocab=pruned_vocab, merges=pruned_merges, unk_token="<unk>"))
litetoken_core.normalizer = normalizers.Sequence([normalizers.NFKC()])
litetoken_core.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)
litetoken_core.decoder = decoders.ByteLevel()

litetoken_core.save(str(LITETOKEN_OUTPUT_DIR / "tokenizer.json"))

litetoken_tokenizer = PreTrainedTokenizerFast(
    tokenizer_file=str(LITETOKEN_OUTPUT_DIR / "tokenizer.json"),
    unk_token="<unk>",
    pad_token="<pad>",
    cls_token="<s>",
    sep_token="</s>",
    mask_token="<mask>",
    bos_token="<s>",
    eos_token="</s>"
 )

print("litetoken_original_vocab", len(base_vocab))
print("litetoken_pruned_vocab", len(litetoken_tokenizer))
print("litetoken_removed_tokens", len(imr_set))

litetoken_original_vocab 50000
litetoken_pruned_vocab 47918
litetoken_removed_tokens 2082


## Model (embedding) training and comparison

In [93]:
tokenizers_to_compare = {
    "dropout_bpe_corrupted": dropout_bpe_tokenizer,
    "sentencepiece_unigram_corrupted": sentencepiece_tokenizer,
    "litetoken_pruned_corrupted": litetoken_tokenizer
}

list(tokenizers_to_compare.keys())

['dropout_bpe_corrupted',
 'sentencepiece_unigram_corrupted',
 'litetoken_pruned_corrupted']

In [94]:
from difflib import SequenceMatcher

def tokenizer_intrinsic_metrics(tokenizer_obj, clean_texts, noisy_texts, max_samples=80_000):
    n = min(len(clean_texts), len(noisy_texts), max_samples)
    clean_subset = clean_texts[:n]
    noisy_subset = noisy_texts[:n]

    total_chars = 0
    total_tokens = 0
    total_words = 0
    total_token_count_for_fertility = 0
    split_words = 0
    stability_scores = []

    for clean_text, noisy_text in tqdm(zip(clean_subset, noisy_subset), total=n, desc=f"Evaluating"):
        clean_text = str(clean_text)
        noisy_text = str(noisy_text)

        enc_clean = tokenizer_obj(clean_text, add_special_tokens=False)
        enc_noisy = tokenizer_obj(noisy_text, add_special_tokens=False)

        clean_ids = enc_clean["input_ids"]
        noisy_ids = enc_noisy["input_ids"]

        total_chars += len(clean_text)
        total_tokens += len(clean_ids)

        words = clean_text.split()
        total_words += len(words)
        for word in words:
            word_ids = tokenizer_obj(word, add_special_tokens=False)["input_ids"]
            total_token_count_for_fertility += len(word_ids)
            if len(word_ids) > 1:
                split_words += 1

        if len(clean_ids) == 0 and len(noisy_ids) == 0:
            stability_scores.append(1.0)
        else:
            stability_scores.append(SequenceMatcher(a=clean_ids, b=noisy_ids).ratio())

    compression_ratio = total_chars / total_tokens if total_tokens > 0 else 0.0
    fertility = total_token_count_for_fertility / total_words if total_words > 0 else 0.0
    fragmentation_rate = split_words / total_words if total_words > 0 else 0.0
    stability_under_noise = float(np.mean(stability_scores)) if stability_scores else 0.0

    return {
        "compression_ratio": float(compression_ratio),
        "fertility": float(fertility),
        "fragmentation_rate": float(fragmentation_rate),
        "stability_under_noise": float(stability_under_noise),
        "samples_used": int(n)
    }

intrinsic_tokenizers = {
    "baseline_roberta": base_tokenizer,
    **tokenizers_to_compare
}

clean_texts = train_data["text"].astype(str).tolist()
noisy_texts = train_data_corrupted["text"].astype(str).tolist()

intrinsic_results = []
for tokenizer_name, tokenizer_obj in intrinsic_tokenizers.items():
    metrics = tokenizer_intrinsic_metrics(
        tokenizer_obj=tokenizer_obj,
        clean_texts=clean_texts,
        noisy_texts=noisy_texts,
        max_samples=80_000
    )
    intrinsic_results.append({"tokenizer": tokenizer_name, **metrics})

intrinsic_results_df = pd.DataFrame(intrinsic_results).sort_values(
    ["stability_under_noise", "compression_ratio"],
    ascending=[False, False]
).reset_index(drop=True)

os.makedirs("results", exist_ok=True)
intrinsic_results_df.to_csv("results/intrinsic_tokenizer_results.csv", index=False)

intrinsic_results_df

Evaluating: 100%|██████████| 80000/80000 [01:24<00:00, 950.98it/s] 



,tokenizer,compression_ratio,fertility,fragmentation_rate,stability_under_noise,samples_used
0,litetoken_pruned_corrupted,3.365199,1.841775,0.476550,0.352003,80000
1,baseline_roberta,4.537492,1.621346,0.390179,0.263933,80000
2,sentencepiece_unigram_corrupted,4.729036,1.305186,0.230535,0.254966,80000
3,dropout_bpe_corrupted,3.971127,1.558367,0.355410,0.231183,80000


In [96]:
def run_tokenizer_experiment(tokenizer_name, tokenizer_obj, num_epochs=3, max_train_samples=20000):
    train_frame = train_data_corrupted.copy()
    if max_train_samples is not None and max_train_samples < len(train_frame):
        train_frame = train_frame.sample(n=max_train_samples, random_state=42).reset_index(drop=True)

    train_ds = TextClassificationDataset(train_frame, tokenizer_obj, max_length=128)
    eval_ds = TextClassificationDataset(test_data_corrupted, tokenizer_obj, max_length=128)

    model_local = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=4
    ).to(device)
    model_local.resize_token_embeddings(len(tokenizer_obj))

    for param in model_local.roberta.encoder.parameters():
        param.requires_grad = False
    for param in model_local.roberta.embeddings.parameters():
        param.requires_grad = True
    for param in model_local.classifier.parameters():
        param.requires_grad = True

    trainable_params = sum(p.numel() for p in model_local.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model_local.parameters())

    output_dir = f"tokenizer_compare/{tokenizer_name}"

    local_args = TrainingArguments(
        output_dir=output_dir,
        learning_rate=1e-4,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=32,
        num_train_epochs=num_epochs,
        weight_decay=0.01,
        logging_steps=100,
        save_strategy="epoch",
        eval_strategy="epoch",
        report_to="none",
        fp16=torch.cuda.is_available()
    )

    local_trainer = Trainer(
        model=model_local,
        args=local_args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer_obj),
        compute_metrics=compute_metrics
    )

    if tokenizer_name != "baseline_roberta":
        local_trainer.train()
        local_trainer.save_model(output_dir)
        tokenizer_obj.save_pretrained(output_dir)
    else:
        print(f"Evaluating baseline with original tokenizer...")

    pred = local_trainer.predict(eval_ds)
    preds = pred.predictions.argmax(axis=-1)
    labels = pred.label_ids

    return {
        "tokenizer": tokenizer_name,
        "test_accuracy": float(accuracy_score(labels, preds)),
        "test_macro_f1": float(f1_score(labels, preds, average="macro")),
        "trainable_params": int(trainable_params),
        "total_params": int(total_params),
        "trainable_ratio": float(trainable_params / total_params),
        "saved_model_dir": output_dir
    }

In [97]:
comparison_results = []

tokenizers_to_compare = {
    "baseline_roberta": base_tokenizer,
    **tokenizers_to_compare
}
for tokenizer_name, tokenizer_obj in tokenizers_to_compare.items():
    result = run_tokenizer_experiment(
        tokenizer_name=tokenizer_name,
        tokenizer_obj=tokenizer_obj,
        num_epochs=3,
        max_train_samples=80000
    )
    comparison_results.append(result)

comparison_results_df = pd.DataFrame(comparison_results).sort_values("test_macro_f1", ascending=False)

os.makedirs("results", exist_ok=True)
comparison_results_df.to_csv("results/tokenizer_comparison_results.csv", index=False)

comparison_results_df

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: textattack/roberta-base-AG-news
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Evaluating baseline with original tokenizer...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: textattack/roberta-base-AG-news
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

,tokenizer,test_accuracy,test_macro_f1,trainable_params,total_params,trainable_ratio,saved_model_dir
3,litetoken_pruned_corrupted,0.714211,0.710746,37791748,122846212,0.307635,tokenizer_compare/litetoken_pruned_corrupted
1,dropout_bpe_corrupted,0.685395,0.686835,39390724,124445188,0.316531,tokenizer_compare/dropout_bpe_corrupted
2,sentencepiece_unigram_corrupted,0.668684,0.669315,39390724,124445188,0.316531,tokenizer_compare/sentencepiece_unigram_corrupted
0,baseline_roberta,0.562895,0.563574,39594244,124648708,0.317647,tokenizer_compare/baseline_roberta


# Results

## LiteToken
+ LiteToken emerged as the most robust approach, achieving the highest Test Accuracy (0.714) and Stability Under Noise (0.352).
+ The Fragmentation Paradox: LiteToken shows the highest Fertility (1.84) and Fragmentation Rate (0.476). While high fragmentation is often viewed negatively, here it acts as a feature: by breaking words into smaller, more frequent "canonical" pieces, the model is less sensitive to single-character OCR errors. But this claim needs to be investigated more
+ Stability Increment: The stability score is nearly 33% higher than the baseline. This suggests that the pruning process effectively removed rare, noisy tokens that usually "trick" the model during inference.

## SentencePiece

+ Highest Compression: SentencePiece has the highest Compression Ratio (4.72) and lowest Fertility (1.30). It represents text using the fewest tokens possible.

+ Accuracy increment: Despite its efficiency, its Accuracy (0.668) is significantly lower than LiteToken. I assume, because Unigram prefers longer subwords, a single OCR error in a long token completely changes its ID, providing no "partial" information to the model

## BPE-Dropout

+ Some of noisy tokens were dropped, so slight increment in intrinsic and extrinsic metrics was achived comapred to baseline tokenizer.


# Conclusion

The results of this study demonstrate that tokenization can be a critical bottleneck for NLP models operating on noisy, real-world data like OCR-scanned documents. This study shows that applying tokenization techniques can enhance results on downstream tasks.



# Limitations
+ Pretrained Domain Shift: Models are pretrained on "perfect" text corpora. Introducing OCR corruptions creates a significant distribution shift from the data the model expects, potentially causing inconsistent performance or hallucinated outputs.  

+ Underlearning Risk: Pruning or modifying a tokenizer requires the model to re-map its learned embeddings. There is a risk that rare but important subwords won't receive enough training signal, leading to poor semantic representation.  

# Future work


+ Character-Based Models: Exploring "token-free" architectures (e.g., ByT5) to learn directly from bytes or characters, bypassing subword fragmentation issues entirely.  

+ Error-Aware Tokenization: Developing tokenizers that use visual similarity (e.g., '1' vs. 'l') to group noisy variations into single canonical tokens.  

+ Visual-Textual Hybrids: Incorporating raw document image patches as auxiliary inputs to help the model resolve textual ambiguities where OCR failed.  

+ Dynamic Regularization: Implementing more complex subword regularization that adjusts merge probabilities based on character-level confidence scores. 
